In [1]:
import pandas as pd
# Install spacy
## -q is for "quiet". It removes logs
%pip install spacy -q
# Import spacy, if you don't have it --> !pip install spacy
import spacy
spacy.__version__
# Download model
!python -m spacy download en_core_web_sm -q
# Import english stop words
from spacy.lang.en.stop_words import STOP_WORDS

Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
data = pd.read_csv('data.csv')

In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load() 
docs = list(nlp.pipe(data['description'].astype(str)))
for doc in docs:

# Print tokens
    print("Tokens: {}".format([token.text for token in doc]))
    print("------------")

    # Print Lemmatized tokens
    print("Lemma: {}".format([token.lemma_ for token in doc]))
    print("------------")

    # Print Indices of each tokens
    print("Indices: {}".format([token.i for token in doc]))
    print("------------")

    # Print whether tokens are alpha numeric
    print("Alpha Numeric: {}".format([token.is_alpha for token in doc]))
    print("------------")

    # Print whether tokens are numbers
    print("Numbers: {}".format([token.like_num for token in doc]))

# Ajouter des stop words personnalisés
nlp.Defaults.stop_words.update({"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"})

# Appliquer les modifications au vocabulaire
for word in {"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"}:
    nlp.vocab[word].is_stop = True

In [4]:
# Print stop words
print(STOP_WORDS)

{"'re", 'via', 'serious', 'be', 'his', 'nine', 'without', 'whereupon', 'ourselves', 'does', 'if', 'the', 'therein', 'all', 'ours', 'meanwhile', 'yourselves', 'no', 'various', 'nothing', 'then', 'myself', 'not', 'more', 'others', 'something', 'around', 'on', 'perhaps', 'hence', 'could', 'toward', 'become', 'became', 'moreover', 'much', 'where', 'none', 'whence', 'from', 'least', 'most', 'done', 'someone', 'a', 'has', 'few', 'and', 'everything', '‘ll', 'hereafter', 'however', '’d', 'namely', '’re', 'go', 'becomes', 'seemed', 'other', 'which', 'twenty', 'side', 'own', 'bottom', 'out', 'so', 'put', 'down', 'whom', '’ve', 'everywhere', 'whoever', 'back', 'been', 'anyone', 'them', 'being', 'beforehand', 'during', 'thereby', 'will', 'upon', 'alone', 'are', 'may', 'they', 'might', 'once', 'must', 'is', 'beyond', 'sixty', '’ll', 'mostly', 'take', 'whereafter', 'again', '‘re', 'i', 'can', 'forty', 'among', 'anything', 'becoming', 'wherever', 'move', 'sometime', 'himself', 'though', 'anywhere', '

In [5]:
# Install wordcloud if necessary
# %pip install wordcloud -q

In [ ]:
import re
# Import matplotlib and wordcloud
import matplotlib.pyplot as plt
import wordcloud
from spacy.lang.en.stop_words import STOP_WORDS

text = " ".join(data['description'].astype(str))
text = re.sub(r"<[^>]+>", " ", text) # replace html
text = re.sub(r"[!\"#$%&()*+,-./:;<=>?@[\\\]^_`{|}~]+", " ", text).lower() # lowercase and punctuation
text = re.sub(r" {2,}", " ", text).strip(" ") # remove concatened spaces and spaces before and after strings

doc = nlp(text)
lemma = [token.lemma_ for token in doc if token.lemma_ not in STOP_WORDS]
lemma

cloud = wordcloud.WordCloud(
    stopwords=STOP_WORDS, 
    collocations=False, 
    min_font_size=6,
    background_color="white", 
    colormap="viridis"
).generate(" ".join(lemma)) # remove bigrams and stopwords

# Show word cloud with matplotlib
plt.imshow(cloud)
plt.axis("off")

# Remove trailing logs
plt.show()

TypeError: expected string or bytes-like object, got 'list'

In [ ]:
from collections import Counter; Counter(lemma).most_common(30)

La taille d’un mot dépend de sa fréquence dans le texte : plus il est gros, plus il apparaît souvent. La position, l’orientation et la couleur sont aléatoires et n’ont aucune signification. dwr signifie Durable Water Repellent. g pourrait mesurer le poids. 

Prochaines étapes 
puis data mining avec tf idf
retirer le bruit

In [ ]:
# # import displacy
# from spacy import displacy

# # render displacy
# sample = nlp(data['description'].astype(str).iloc[0])
# displacy.render(list(sample.sents)[0], style='dep', jupyter=True,options={'distance': 70})

# text mining

## create an array of odcuments

In [ ]:
type(lemma)

In [ ]:
import pandas as pd
series = pd.Series(lemma)

In [ ]:
series = pd.Series([" ".join(token.lemma_ for token in d if token.lemma_ not in STOP_WORDS) for d in docs])
df = pd.DataFrame(series, columns=["catalogue"])

In [ ]:
df.value_counts() > 50

In [ ]:
type(df)

In [ ]:
df.head()

In [ ]:
# Tfidf transformer
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF vector
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['catalogue'])

# X is a generator. We can transform that as an array
dense = X.toarray()
dense


In [ ]:
# Creating a DataFrame with all the words and tf-idf score
tfid_df = pd.DataFrame(dense,
                        columns=[x for x in vectorizer.get_feature_names_out()])

tfid_df

In [ ]:
# Transpose dataframe
tfid_df = tfid_df.T

# Create a new token column
tfid_df["token"] = tfid_df.index

tfid_df

In [ ]:
# Sort the top 15 values by descending order
tfid_df.sort_values(by=0, ascending=False)[:15]

In [ ]:
# Import plotly express
import plotly.express as px

# Create bar chart
fig = px.bar(tfid_df.sort_values(by=0, ascending=False)[:15], x="token", y=0)

# Render chart
fig.show()